In [1]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score


# Feature selection
def rfeFeature(indep_X, dep_Y, n):

    rfelist = []
    selected_features = []

    models = [
        LinearRegression(),
        SVR(kernel="linear"),
        DecisionTreeRegressor(random_state=0),
        RandomForestRegressor(n_estimators=10, random_state=0)
    ]

    for model in models:

        rfe = RFE(estimator=model, n_features_to_select=n)

        X_rfe = rfe.fit_transform(indep_X, dep_Y)

        rfelist.append(X_rfe)

        feature = indep_X.columns[rfe.support_]

        selected_features.append(feature)

    return rfelist, selected_features


# Train test split + scaling
def split_scaler(indep_X, dep_Y):

    X_train, X_test, Y_train, Y_test = train_test_split(
        indep_X,
        dep_Y,
        test_size=0.25,
        random_state=0
    )

    sc = StandardScaler()

    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)

    return X_train, X_test, Y_train, Y_test


# Prediction metrics
def r2_prediction(regressor, X_test, y_test):

    y_pred = regressor.predict(X_test)

    r2 = r2_score(y_test, y_pred)

    return r2


# Linear Regression
def linear(X_train, Y_train, X_test, Y_test):

    regressor = LinearRegression()

    regressor.fit(X_train, Y_train)

    r2 = r2_prediction(regressor, X_test, Y_test)

    return r2


# SVM Linear
def svm_linear(X_train, y_train, X_test, y_test):

    regressor = SVR(kernel='linear')

    regressor.fit(X_train, y_train)

    r2 = r2_prediction(regressor, X_test, y_test)

    return r2


# SVM Non Linear
def SVM_NL(X_train, Y_train, X_test, Y_test):

    regressor = SVR(kernel="rbf")

    regressor.fit(X_train, Y_train)

    r2 = r2_prediction(regressor, X_test, Y_test)

    return r2


# Decision Tree
def Decision(X_train, y_train, X_test, y_test):

    regressor = DecisionTreeRegressor(random_state=0)

    regressor.fit(X_train, y_train)

    r2 = r2_prediction(regressor, X_test, y_test)

    return r2


# Random Forest
def random(X_train, y_train, X_test, y_test):

    regressor = RandomForestRegressor(
        n_estimators=10,
        random_state=0
    )

    regressor.fit(X_train, y_train)

    r2 = r2_prediction(regressor, X_test, y_test)

    return r2


# Final Result DataFrame
def ref_regression(
        acclog,
        accsvml,
        accsvmnl,
        accdes,
        accrf):

    rfedataframe = pd.DataFrame(
        index=['Linear', 'SVMl', 'Decision', 'Random'],
        columns=['Linear', 'SVMl', 'SVMnl', 'Decision', 'Random']
    )

    for number, idx in enumerate(rfedataframe.index):

        rfedataframe.loc[idx, 'Linear'] = acclog[number]
        rfedataframe.loc[idx, 'SVMl'] = accsvml[number]
        rfedataframe.loc[idx, 'SVMnl'] = accsvmnl[number]
        rfedataframe.loc[idx, 'Decision'] = accdes[number]
        rfedataframe.loc[idx, 'Random'] = accrf[number]

    return rfedataframe


# Load Dataset
dataset1 = pd.read_csv("prep.csv")

df2 = pd.get_dummies(dataset1, drop_first=True)

indep_X = df2.drop('classification_yes', axis=1)

dep_Y = df2['classification_yes']


# RFE Features
rfelist, selected_features = rfeFeature(indep_X, dep_Y, 3)

acclog = []
accsvml = []
accsvmnl = []
accdes = []
accrf = []


# Training Loop
for i in rfelist:

    X_train, X_test, y_train, y_test = split_scaler(i, dep_Y)

    acclog.append(
        linear(X_train, y_train, X_test, y_test)
    )

    accsvml.append(
        svm_linear(X_train, y_train, X_test, y_test)
    )

    accsvmnl.append(
        SVM_NL(X_train, y_train, X_test, y_test)
    )

    accdes.append(
        Decision(X_train, y_train, X_test, y_test)
    )

    accrf.append(
        random(X_train, y_train, X_test, y_test)
    )


# Final Result
result = ref_regression(
    acclog,
    accsvml,
    accsvmnl,
    accdes,
    accrf
)

print(result)



            Linear      SVMl     SVMnl  Decision    Random
Linear    0.441961  0.262153  0.262162  0.441961  0.441816
SVMl      0.441961  0.262153  0.262162  0.441961  0.441816
Decision  0.664893  0.609652  0.883134  0.965961  0.916304
Random    0.676174  0.670691  0.900941  0.933504  0.887256
